# Stack Observations

This is an extension of the [multi-observation tutorial](multiresolution) (with an identical setup) and shows how to stack observations from multiple observations get improve the computational performance of the fit.

In [ ]:
# Import Packages
import astropy.io.fits as fits
import astropy.units as u
import jax.numpy as jnp
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.wcs import WCS

import scarlet2 as sc2
sc2.set_validation(False)

## Load Data

We first load the HSC and HST images, PSFs and weight/variance maps and store them in their own {py:class}`~scarlet2.Observation` instances. We also load a catalog of sources detected jointly from the observations (see [here](https://github.com/astro-data-lab/scarlet-test-data/blob/main/scarlet_test_data/data/multiresolution_tutorial/get_source_catalog.py) for details on how this catalog was created).

In [ ]:
from huggingface_hub import hf_hub_download

filename = hf_hub_download(
    repo_id="astro-data-lab/scarlet-test-data",
    filename="multiresolution_tutorial/data.fits.gz",
    repo_type="dataset",
)
with fits.open(filename) as hdul:
    # Load two Observations
    obs_hst = sc2.Observation(
        # Data amplitude is ~10x larger in HST as in HSC
        # For better optimization: adjust data and variance
        # details at https://github.com/pmelchior/scarlet2/issues/240
        hdul["HST_OBS"].data / 10,
        weights=hdul["HST_WEIGHTS"].data * 100,
        psf=hdul["HST_PSF"].data,
        wcs=WCS(hdul["HST_OBS"].header),
        channels=["F814W"],
        name="HST",
    )
    obs_hsc = sc2.Observation(
        hdul["HSC_OBS"].data,
        weights=hdul["HSC_WEIGHTS"].data,
        psf=hdul["HSC_PSF"].data,
        channels=["g", "r", "i", "z", "y"],
        wcs=WCS(hdul["HSC_OBS"].header),
        name="HSC",
    )

    # Load source catalog
    coords_table = Table(hdul["CATALOG"].data)
    radecsys = hdul["CATALOG"].header["RADECSYS"]
    equinox = hdul["CATALOG"].header["EQUINOX"]
    ra_dec = SkyCoord(
        ra=coords_table["RA"] * u.deg,
        dec=coords_table["DEC"] * u.deg,
        frame=radecsys.lower(),
        equinox=f"J{equinox}",
    )

Now we display the observation contents:

In [ ]:
norm_hst = sc2.plot.AsinhAutomaticNorm(obs_hst)
norm_hsc = sc2.plot.AsinhAutomaticNorm(obs_hsc)

sc2.plot.observation(
    obs_hst,
    norm=norm_hst,
    add_peaks=ra_dec,
    show_psf=True,
    label_kwargs={"color": "red"}
);
sc2.plot.observation(
    obs_hsc,
    norm=norm_hsc,
    add_peaks=ra_dec,
    show_psf=True
);

## Define Model Frame and Stack Observations

As we have two different instruments with different pixel resolutions, we need define a model frame that allows us to jointly model these data:

In [ ]:
model_frame = sc2.Frame.from_observations(
    observations=[obs_hst, obs_hsc],
    coverage="intersection",
)

This method defines a common area for both observations, in this case a box covering the intersection of both. This is the more sensible choice when stacking observations because than every observation has valid pixels in the area that is jointly modelled.

The main difference in this tutorial is that we construct a single observation that treats the different instruments as different channels of the same (virtual) instrument, which happen to have different wavelengths and PSFs, but the same pixel grid, namely that of the model frame. Doing so will require padding or truncation in the observation has the same pixel grid, and resampling if not. That way, the new observation does not require resampling during the fit and all evaluations run in parallel. Because resampling correlates pixel values, we automatically get a {py:class}`~scarlet2.CorrelatedObservation`:

In [ ]:
obs_stacked = sc2.stack_observations([obs_hst, obs_hsc], model_frame)
print(type(obs_stacked).__name__)

For more information on correlated data, see [this](correlated) tutorial.

## Initialize Sources from a Stacked Observation

The initialization of the sources follows our general recommendation (from e.g. the [quickstart guide](../0-quickstart)):

In [ ]:
with sc2.Scene(model_frame) as scene:
    infos = sc2.init.hierarchical_sources(
        obs_hst,
        K=3,
        scales=[2,3,4],
        min_area=16,
        strict=True,
        catalog=ra_dec,
        split_peaks=False,
        image_type="space"
    )
    for sinfo in infos:
        sc2.Source(sinfo.center, sinfo.spectrum, sinfo.morphology)

We choose the HST image for the initialization because it has the higher resolution and should therefore be able to better identify the position and shape of the sources. It shares the same resolution as the model frame, so the morphologies are correctly sized already. We also set `split_peaks=False` because the sources (especially the groups 0-4 and 7-9) definitely overlap, so we want to allow the footprints to overlap as well. And because the space-based PSF wodth is on the order of 1 pixel, the multi-scale support calculation needs to be adjusted with `image_type="space"`.

So far we know nothing about the source spectra in the HSC bands.
But we can use the HST-defined morphology to perform forced photometry in all bands, including the HSC bands. So, we should get a reasonable model across all channels of the stacked observation.

In [ ]:
forced_spectra = sc2.measure.forced_photometry(scene, obs_stacked)
for i in range(len(scene.sources)):
    scene.sources[i] = scene.sources[i].replace("spectrum", forced_spectra[i])

We can have a quick look how the initial model looks like. We use `split_channels=True` so that we don't visually mix the different observations into one (very false) color image.

In [ ]:
sc2.plot.scene(
    scene,
    observation=obs_stacked,
    show_rendered=True,
    show_observed=True,
    show_residual=True,
    add_labels=True,
    add_boxes=True,
    box_kwargs={"edgecolor": "red", "facecolor": "none"},
    label_kwargs={"color": "red"},
    split_channels=True
);

The different resolution of the different channels in this stacked observation is very apparent. But all the pixels are lined up, all channels are fully defined. The only operation that needs to happen for the model to match the data is a convolution per channel, and these are done in parallel. So, let's define the fit parameter and run the fitter:

In [ ]:
from functools import partial
from numpyro.distributions import constraints

# best step size for parameters with a relative "uncertainty":
# big/bright sources adjust more quickly
spec_step = partial(sc2.relative_step, factor=0.05)
morph_step = 0.2

with sc2.Parameters(scene):
    for i in range(len(scene.sources)):
        # because we want the spectrum parameter to be constrained,
        # we need to ensure that their initialization satisfies the constraint
        tiny = 1e-6
        spectrum = scene.sources[i].spectrum
        if (spectrum <= 0).any():
            scene.sources[i] = scene.sources[i].replace("spectrum", jnp.maximum(spectrum, tiny))
        sc2.Parameter(
            scene.sources[i].spectrum,
            name=f"spectrum:{i}",
            constraint=constraints.positive,
            stepsize=spec_step,
        )
        # same for [0,1] constrained morphology
        morph = spectrum = scene.sources[i].morphology
        if ((morph <= 0) | (morph >= 1)).any():
            scene.sources[i] = scene.sources[i].replace("morphology", jnp.clip(morph, tiny, 1-tiny))
        sc2.Parameter(
            scene.sources[i].morphology,
            name=f"morph:{i}",
            constraint=constraints.unit_interval,
            stepsize=morph_step,
        )

In [ ]:
scene_ = scene.fit(obs_stacked, max_iter=1000, progress_bar=True)

We can now compare the model from the stacked observation to the individual observations:

In [ ]:
sc2.plot.scene(
    scene_,
    observation=obs_hst,
    norm=norm_hst,
    show_rendered=True,
    show_observed=True,
    show_residual=True,
    add_labels=True,
    add_boxes=True,
    box_kwargs={"edgecolor": "red", "facecolor": "none"},
    label_kwargs={"color": "red"},
);
sc2.plot.scene(
    scene_,
    observation=obs_hsc,
    norm=norm_hsc,
    show_rendered=True,
    show_observed=True,
    show_residual=True,
    add_labels=True,
    add_boxes=True,
    box_kwargs={"edgecolor": "red", "facecolor": "none"},
    label_kwargs={"color": "red"},
);

The model is pretty good, but the residuals shows more pronounced dipoles as the standard multi-resolution fit. That's is a consequence of the stacking PSFs with different shapes, but we have a mechanism to fix that:

## Fit Astrometric Offsets

The stacked observation has one renderer (which is what makes it fast). The actual transformation is a convolution and has an attribute `.shift` that we can define as a {py:class}`~scarlet2.Parameter` and refit. [Originally](multiresolution), we define the shift for the HSC observation only, which defined it as the relative astrometric correction wrt. the HST observation. With a stacked observation, we can't do that because there's just one pixel grid. But the shift can be set as a per-channel shift. This sounds flexible enought, but is in fact too flexible. If shifts are permitted in all channels, nothing remains to define the astrometric reference.

```{tip}

Define per-channel shifts with a multi-dimensional prior that has essentially zero width for at least one channel. This channel (or sets of channels) then defines the astrometric reference. 
```

In [ ]:
# define shift parameter: unconstrained, with milli-arcsec stepsize
import astropy.units as u
import numpyro.distributions as dist

# replace the 2D shift with a (C,2) shift
C = obs_stacked.frame.C
shift = jnp.zeros((C, 2))
object.__setattr__(obs_stacked.renderer[-1], "shift", shift)

ref_channel = 0
with sc2.Parameters(obs_stacked):
    # define a Gaussian prior with zero mean and narrow width in ref_channel
    scale = jnp.ones((C, 2))
    scale = scale.at[ref_channel].set(1e-3)
    scale = 0.1 * scale * u.arcsec
    prior = dist.Normal(shift, scale=scale).to_event(2) # last 2 dims are per-event

    # declare new shift as parameter
    sc2.Parameter(obs_stacked.renderer[-1].shift, name="shift", prior=prior, stepsize=1e-3 * u.arcsec)

# the more flexible fitter optimizes scene and observation and returns both
scene_, obs_stacked_ = sc2.infer.fit(scene_, obs_stacked, e_rel=1e-4, max_iter=1000)

# get the astrometry shift
shift = obs_stacked_.get("shift")
print("\nShift per channel in model pixels:")
print(shift)

Now we have the shift in every channel as an independent variable, with the first channel clamped down to almost zero. That can be appropriate, in particular if the PSFs in HSC are not well-aligned internally, but usually the problem stems from differences in astrometric fitting and PSF modeling between instruments or surveys. Allowing every channel to have its own shift is overkill and may pick up spurious shifts from noise or complex wavelength-dependent morphologies. We could average all of the HSC-channel shifts, but the best option is to create a shift per observation, where the channels of the original observations share the shift parameter. The following achieves this task:

In [ ]:
class ShiftPerObservation(sc2.Module):
    shifts: tuple
    channel_index: tuple
    C: int

    def __init__(self, observations, frame):
        if not isinstance(observations, (list, tuple)):
            observations = (observations,)
        self.shifts = tuple(jnp.zeros((2,)) for obs in observations)
        self.channel_index = tuple(jnp.array([frame.channels.index(c) for c in obs.frame.channels]) for obs in observations)
        self.C = frame.C
    def __call__(self):
        shift = jnp.zeros((C, 2))
        for i,c in enumerate(self.channel_index):
            shift = shift.at[c].set(self.shifts[i])
        return shift

shifted_obs = (obs_hsc,) # only allow HSC obs to shift
per_obs_shift = ShiftPerObservation(shifted_obs, obs_stacked.frame)
object.__setattr__(obs_stacked.renderer[-1], "shift", per_obs_shift)

with sc2.Parameters(obs_stacked):
    # define a Gaussian prior with zero mean and narrow width in ref_channel
    scale = 0.1 * jnp.ones((2,)) * u.arcsec
    prior = dist.Normal(jnp.zeros((2,)), scale=scale).to_event(1) # last dim are per-event

    # declare new shift(s) as parameter(s)
    for shift,obs in zip(per_obs_shift.shifts, shifted_obs):
        sc2.Parameter(shift, name=f"shift_{obs.name}", prior=prior, stepsize=1e-3 * u.arcsec)

# let's fit again..
scene_, obs_stacked_ = sc2.infer.fit(scene_, obs_stacked, e_rel=1e-4, max_iter=1000)

# get the astrometry shift
shift = obs_stacked_.renderer[-1].shift()
print("\nShift per channel in model pixels:")
print(shift)
hsc_shift = obs_stacked_.get("shift_HSC")
print(f"Shift of HSC in model pixels: {hsc_shift}")

So, all HSC channels share the same shift, and the HST observation didn't shift at all: perfect! To evaluate the model in the original pixel grid of HSC, we need to convert the astrometric offset we have just determined from the model grid:

In [ ]:
# get the Jacobian transform between the resampled and native HSC frame
jacobian, _ = sc2.frame.get_relative_jacobian_shift(obs_stacked.frame, obs_hsc.frame)
shift_native = jacobian @ hsc_shift
print(f"Shift in native HSC pixels: {shift_native}")

# force the renderer to use optimized value for shift
object.__setattr__(obs_hsc.renderer[-1], "shift", shift_native)

In [ ]:
sc2.plot.scene(
    scene_,
    observation=obs_hst,
    show_rendered=True,
    show_observed=True,
    show_residual=True,
    add_labels=True,
    add_boxes=True,
    norm=norm_hst,
    box_kwargs={"edgecolor": "red", "facecolor": "none"},
    label_kwargs={"color": "red"},
)
sc2.plot.scene(
    scene_,
    observation=obs_hsc,
    show_rendered=True,
    show_observed=True,
    show_residual=True,
    add_labels=True,
    add_boxes=True,
    norm=norm_hsc,
);

Look Ma, no dipoles! We can thus concentrate on making the source models better...